In [3]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]



In [12]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [13]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [14]:
# Send the question with the tool attached
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

''

In [15]:
len(response.output)
call = response.output[0]
# LLM returns a tool call decision, not an answer
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration"}', call_id='call_1ksfzBjZZDJUQlWnC9NrkutQ', name='search', type='function_call', id='fc_02346984a03c0fb4006a3a3efdafc88195a81b28d38a4d8a65', namespace=None, status='completed')

In [16]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [17]:
# Parse the arguments and execute the search

import json

args = json.loads(call.arguments)
results = search(**args)
result_json = json.dumps(results, indent=2)
args


{'query': 'join course discovered course can I join enrollment registration'}

In [18]:
# Send results back with full conversation history
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}
messages.append(call)
messages.append(function_call_output)

In [19]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [24]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [27]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join registration enrollment new student"}', call_id='call_FPKMBwte1qOgfoKMzww4KF7A', name='search', type='function_call', id='fc_0540119571616e69006a3a3f21d8608194b22918fa0634c8d8', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment join late registration discovered course"}', call_id='call_FbfpF1eRLnoG4DBlFyRM4YsV', name='search', type='function_call', id='fc_0540119571616e69006a3a3f21d8748194aa11db5016ebb8fa', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I still join the course after it started FAQ"}', call_id='call_4xWcoWKmnm1qevAp9uMLm422', name='search', type='function_call', id='fc_0540119571616e69006a3a3f21d87c81948664aeb3451ecdbb', namespace=None, status='completed')]

In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(166, 90)

In [14]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [30]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [31]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join FAQ"}', call_id='call_KTYXWLKjU1tdhdBgSPUoAEW2', name='search', type='function_call', id='fc_07533d1dc1765f25006a3a3f5b4de88196ab5358aa2f525393', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student join course enrollment FAQ discovered course"}', call_id='call_imiI1JgMFn797yesmvBPIF83', name='search', type='function_call', id='fc_07533d1dc1765f25006a3a3f5b4e048196b108eb568c0e73c4', namespace=None, status='completed')]

In [32]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [33]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
function_call: search {"query":"new student join course enrollment FAQ discovered course"}


In [34]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [35]:
response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions. If you’re just learning for the content, you can start anytime.\n\nIf you want, I can also help with how to get started or explain the certificate requirements.'

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [37]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure to submit your project while the course is still accepting submissions.

If you’d like, I can also help you figure out how to start the course or explain the certificate requirements.


In [40]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                print(item.content[0].text)
        
        it = it + 1
        if has_function_calls == False:
            break
    

In [46]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [39]:
result = agent_loop(instructions, question)

NameError: name 'agent_loop' is not defined

In [38]:
agent_loop(instructions, "How do I run Olama locally?")

NameError: name 'agent_loop' is not defined

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""



In [51]:
question = "what's queen gambit?"
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit queen's gambit chess opening FAQ"}
iteration #2...
function_call: search {"query":"queen gambit chess opening queen's gambit FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any course-FAQ entry about “queen’s gambit,” so it looks like this isn’t a course/logistics question.

If you meant the chess opening, I’d be happy to help with that in a general sense, but I can’t answer it from the course FAQ alone. Is there anything else course-related you want to explore?


In [52]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


In [54]:
def search(query: str)-> dict[str, str]: 
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [55]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [56]:
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [58]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [60]:
# The chat interface and runner
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")  # specify model explicitly
)

In [61]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [62]:
result.cost

CostInfo(input_cost=Decimal('0.00110475'), output_cost=Decimal('0.0014175'), total_cost=Decimal('0.00252225'))

In [64]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,   # pick up where we left off
    callback=callback,
)

-> Response received


-> Response received
